# RM Training — Reward Model for Interview Answer Assistant
**Phase 2: Reward Model** using Qwen2.5-3B-Instruct + QLoRA

Steps:
1. Generate rejected answers (70% Qwen2.5-0.5B + 30% rule-based)
2. Train reward model on preference pairs (chosen vs rejected)
3. Save LoRA adapter to Google Drive

Hardware: T4 GPU (free Colab, 16GB VRAM)
Time: ~3 hours total

In [ ]:
# @title 1. Install Dependencies & Clone Repo
!pip install -q transformers peft bitsandbytes trl datasets accelerate pandas tqdm
!git clone https://github.com/kinjazA/RL.git /content/RL 2>/dev/null
%cd /content/RL
!ls data/

In [ ]:
# @title 2. GPU Check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Step 1 — Generate Rejected Answers

In [ ]:
# @title 3. Generate RM Preference Pairs (gen_reject.py)
# Temporarily add repo root to path so imports work
import sys
sys.path.insert(0, "/content/RL")

# Run gen_reject with model generation
!python rm/gen_reject.py --batch_size 4 --model_ratio 0.7

# Verify output
import pandas as pd
df = pd.read_csv("data/rm_train.csv", encoding="utf-8-sig")
print(f"\nRM pairs: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\n--- Sample ---")
row = df.iloc[0]
print(f"Prompt:   {row['prompt'][:120]}")
print(f"Chosen:   {row['chosen'][:150]}...")
print(f"Rejected: {row['rejected'][:150]}...")

## Step 2 — Train Reward Model

In [ ]:
# @title 4. Train Reward Model (train_rm.py)
!python rm/train_rm.py

## Step 3 — Save to Google Drive

In [ ]:
# @title 5. Save RM Adapter to Google Drive
from google.colab import drive
drive.mount("/content/drive")

!cp -r /content/RL/rm/rm_adapter /content/drive/MyDrive/RL_rm_adapter
print("Saved to Google Drive: MyDrive/RL_rm_adapter")
!ls /content/drive/MyDrive/RL_rm_adapter

---
### Done

Outputs:
- `data/rm_train.csv` — preference pairs for RM training
- `rm/rm_adapter/` — trained LoRA weights (also backed up to Drive)

Next: Phase 3 — PPO alignment using the RM